# Game Simulator
Forecast DAU, payer DAU, and revenue over a 365-day horizon by adjusting UA spend, CPI, retention, and conversion inputs.

In [1]:
# show
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import date

from common_lib.sql import BigQueryConnector
from common_lib.sheets import load_inputs, get_inputs_dir
from common_lib.curves import build_curve, anchors_from_df
from common_lib.simulation import (
    SimulationEngine, PlatformInputs,
    save_scenario, load_scenario, list_scenarios,
    save_result, load_result, list_results,
)
from common_lib.widgets import ScenarioPanel

print('Inputs dir:', get_inputs_dir())

Inputs dir: /Users/ivanaguilar/Desktop/DataStuff/gitrepos/testrepo/simulator/config/inputs


## 1. Load baseline actuals from BigQuery

In [2]:
# show
bqc = BigQueryConnector()

actuals_params = {'start_date': '2024-01-01'}
cost_info = bqc.print_cost_estimate('./sql/actuals.sql', is_path=True, query_parameters=actuals_params)

This query will process 7.67 GB when run.
Estimated query cost: $0.05


In [3]:
PLATFORM_MAP = {'AND': 'android', 'IOS': 'ios'}

refresh_data = False  # Set to True to refresh data from BigQuery, False to load from local pickle 

if refresh_data == True:
    actuals = bqc.get('./sql/actuals.sql', is_path=True, query_parameters=actuals_params)
    pd.to_pickle(actuals, './data/actuals_dau.pkl')
else:
    actuals = pd.read_pickle('./data/actuals_dau.pkl')

actuals['dt'] = pd.to_datetime(actuals['dt'])
actuals['platform'] = actuals['platform'].map(PLATFORM_MAP).fillna(actuals['platform'].str.lower())
actuals = actuals.sort_values('dt')

# Anchor DAU: last observed day per platform
anchor_dau = actuals.sort_values('dt').groupby('platform')['dau'].last().to_dict()
print('Anchor DAU:', anchor_dau)

Anchor DAU: {'android': 35491, 'ios': 50162}


In [4]:
# show
# Fetch live retention and conversion curves
cohort_params = {'start_date': '2024-01-01'}
cost_info = bqc.print_cost_estimate('./sql/retention.sql', is_path=True, query_parameters=cohort_params)
cost_info = bqc.print_cost_estimate('./sql/conversion.sql', is_path=True, query_parameters=cohort_params)

This query will process 4.34 GB when run.
Estimated query cost: $0.03
This query will process 4.36 GB when run.
Estimated query cost: $0.03


In [5]:
refresh_data = False  # Set to True to refresh data from BigQuery, False to load from local pickle  

if refresh_data == True:
    live_retention  = bqc.get('./sql/retention.sql',  is_path=True, query_parameters=cohort_params)
    live_retention.to_pickle('./data/live_retention.pkl')
    live_conversion = bqc.get('./sql/conversion.sql', is_path=True, query_parameters=cohort_params)
    live_conversion.to_pickle('./data/live_conversion.pkl')
else:
    live_retention  = pd.read_pickle('./data/live_retention.pkl')
    live_conversion = pd.read_pickle('./data/live_conversion.pkl')

live_retention['platform']  = live_retention['platform'].map(PLATFORM_MAP).fillna(live_retention['platform'].str.lower())
live_conversion['platform'] = live_conversion['platform'].map(PLATFORM_MAP).fillna(live_conversion['platform'].str.lower())

## 2. Load scenario inputs from local CSVs

In [6]:
# show
sheet_inputs = load_inputs()

for name, df in sheet_inputs.items():
    print(f'\n--- {name} ---')


--- retention ---

--- conversion ---

--- ua_spend ---

--- cpi ---

--- arpdau ---


## 3. Build curves

In [7]:
# show
# Build curves from CSV for the preview chart below
curves = {
    platform: {
        'retention':  build_curve(anchors_from_df(sheet_inputs['retention'],  platform)),
        'conversion': build_curve(anchors_from_df(sheet_inputs['conversion'], platform)),
    }
    for platform in ('ios', 'android')
}

In [8]:
# show
# Preview interpolated curves
dx = list(range(1, 366))
fig = make_subplots(rows=1, cols=2, subplot_titles=['Retention D1–D365', 'Conversion D1–D365'])
colors = {'ios': '#007AFF', 'android': '#34C759'}
for platform in ('ios', 'android'):
    c = colors[platform]
    fig.add_trace(go.Scatter(x=dx, y=curves[platform]['retention'],
                             name=f'{platform} retention', line=dict(color=c)), row=1, col=1)
    fig.add_trace(go.Scatter(x=dx, y=curves[platform]['conversion'],
                             name=f'{platform} conversion', line=dict(color=c, dash='dash')), row=1, col=2)
fig.update_layout(height=450, margin=dict(t=40, b=30))
fig.show()

## 4. Interactive Scenario Panel

In [9]:
# show
from common_lib.app import prefill_panel, setup_callbacks

engine = SimulationEngine()
panel  = ScenarioPanel(saved_scenarios=list_scenarios())
prefill_panel(panel, actuals, anchor_dau, sheet_inputs, anchors_from_df)
setup_callbacks(panel, engine, actuals)
panel.display()

## 5. Plot results

`plot(scenarios, chart)` loads saved results from disk and renders charts.

- **`scenarios`** — a single name or list of names (must have been simulated first)
- **`chart`** — `'all'` (default) · `'dau'` · `'installs'` · `'revenue'` · `'payers'` · `'monthly'`

In [ ]:
from common_lib.plots import plot, plot_retention, plot_conversion, configure as configure_plots
from common_lib.simulation import list_results

configure_plots(actuals)

# ── Usage ──────────────────────────────────────────────────────────────────
# plot('base_case')                         # all charts
# plot('base_case', chart='dau')            # DAU only
# plot('base_case', chart='revenue')        # daily revenue
# plot('base_case', chart='monthly')        # monthly bar
# plot(['base_case', 'high_ua'])            # compare scenarios
#
# plot_retention('base_case')               # retention curve from saved scenario
# plot_conversion('base_case')              # conversion curve from saved scenario
# plot_retention(['base_case', 'high_ua'])  # compare retention curves across scenarios
# plot_retention(panel.get_curve_anchors()) # preview current panel state
print('Available results:', list_results())

In [15]:
plot('test3', chart='dau')

In [16]:
plot('test3', chart='revenue')

## 6. Summary table

In [13]:
# show
def summary_table(scenarios=None) -> pd.DataFrame:
    """
    Summarise saved simulation results.
    scenarios: list of names, or None to include all saved results.
    """
    names = scenarios if scenarios is not None else list_results()
    rows = []
    for name in names:
        df = load_result(name)
        for platform in ('ios', 'android', 'combined'):
            sub = df[df['platform'] == platform]
            if sub.empty:
                continue
            rows.append({
                'scenario':       name,
                'platform':       platform,
                'avg_dau':        round(sub['dau'].mean()),
                'peak_dau':       round(sub['dau'].max()),
                'total_installs': round(sub['new_installs'].sum()),
                'total_iap_rev':  round(sub['iap_revenue'].sum(), 2),
                'total_ad_rev':   round(sub['ad_revenue'].sum(), 2),
                'total_revenue':  round(sub['total_revenue'].sum(), 2),
            })
    return pd.DataFrame(rows)


summary_table()

,scenario,platform,avg_dau,peak_dau,total_installs,total_iap_rev,total_ad_rev,total_revenue
0,test3,ios,39910,56437,345133,7093945.48,1734075.56,8828021.04
1,test3,android,27556,39898,230088,4898113.32,1197316.59,6095429.91
2,test3,combined,67466,96335,575221,11992058.80,2931392.15,14923450.95
